# Multinomial Naive Bayes: implementation and reference comparison
This six-sentence toy dataset exercises the code; it cannot establish sentiment-classification quality.
The split leaves only **four training examples and two test examples**. Both implementations use
lowercase word tokens of at least two characters, additive smoothing, and a vocabulary learned
from training text only. No parameters are selected using the test examples.

In [1]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/nb_scratch.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from within its repository folder.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report
from src.nb_scratch import MultinomialNBScratch

data = pd.read_csv(ROOT / 'data/sample_tiny.csv')
if data[['text', 'label']].isna().any().any():
    raise ValueError('Text and labels must not be missing.')
train_text, test_text, train_y, test_y = train_test_split(
    data['text'], data['label'], test_size=0.3, random_state=42, stratify=data['label']
)
print(f'Training examples: {len(train_y)}; test examples: {len(test_y)}')
display(data)

Training examples: 4; test examples: 2


,text,label
0,this movie was wonderful with great acting,pos
1,absolutely fantastic film i loved every minute,pos
2,the plot was boring and predictable,neg
3,terrible acting and a very slow story,neg
4,a delightful and charming performance,pos
5,waste of time not worth watching,neg


In [3]:
reference = make_pipeline(CountVectorizer(), MultinomialNB(alpha=1.0))
reference.fit(train_text, train_y)
scratch = MultinomialNBScratch(alpha=1.0).fit(train_text, train_y)

scratch_log_probs = scratch.predict_log_proba(test_text)
np.testing.assert_array_equal(scratch.classes_, reference.classes_)
np.testing.assert_allclose(scratch_log_probs, reference.predict_log_proba(test_text), atol=1e-12)
np.testing.assert_allclose(np.exp(scratch_log_probs).sum(axis=1), 1.0)
print('Normalized probabilities agree with scikit-learn on the same tokenization.')
results = []
for name, model in [('NumPy scratch', scratch), ('scikit-learn', reference)]:
    prediction = model.predict(test_text)
    results.append({'implementation': name, 'toy_test_accuracy': accuracy_score(test_y, prediction)})
    print(name)
    print(classification_report(test_y, prediction, zero_division=0))
display(pd.DataFrame(results))

Normalized probabilities agree with scikit-learn on the same tokenization.
NumPy scratch
              precision    recall  f1-score   support

         neg       0.50      1.00      0.67         1
         pos       0.00      0.00      0.00         1

    accuracy                           0.50         2
   macro avg       0.25      0.50      0.33         2
weighted avg       0.25      0.50      0.33         2

scikit-learn
              precision    recall  f1-score   support

         neg       0.50      1.00      0.67         1
         pos       0.00      0.00      0.00         1

    accuracy                           0.50         2
   macro avg       0.25      0.50      0.33         2
weighted avg       0.25      0.50      0.33         2



,implementation,toy_test_accuracy
0,NumPy scratch,0.5
1,scikit-learn,0.5


The test set is too small for a meaningful performance comparison. The useful check here is
numerical agreement with the reference. Unknown words are ignored; a document with no known
words receives the learned class priors. This dense implementation is intended for small examples.
A larger labeled dataset with independent evaluation would be needed for a practical sentiment model.